# Notebook 3 — Train / validation / test split

**Job of this notebook:** split the data *before* any deep analysis, so
the test set can't leak into decisions made in EDA or feature engineering.

**Reads:** `data/interim/labeled_table.parquet` (Notebook 2's artifact).
**Writes:** `data/processed/{train,val,test}.parquet`.


In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
from config import LABELED_TABLE_PATH, TRAIN_PATH, VAL_PATH, TEST_PATH, LABEL_COL, RANDOM_STATE

labeled = pd.read_parquet(LABELED_TABLE_PATH)
labeled["order_purchase_timestamp"] = pd.to_datetime(labeled["order_purchase_timestamp"])
labeled.shape


## 1. Random vs. time-based split — which one, and why

Two honest options here:

- **Random split**, stratified on the label. Simple, and keeps the class
  ratio identical across splits. But if delivery performance drifts over
  time (holiday season, courier changes, COVID-era slowdowns, etc.), a
  random split lets the model see future patterns during training that
  it wouldn't have in production.
- **Time-based split** (earliest orders → train, latest → test). Matches
  how the model will actually be used in production — it always predicts
  forward in time from what it's been trained on. The cost is that the
  label ratio can differ across splits if delivery performance itself
  trends over time.

Since the production pipeline will always be scoring *new, future* orders
against a model trained on *past* orders, we split **by time**. A random
split would let the model implicitly learn from data it wouldn't have
access to at prediction time in the real system.


## 2. Check the date range before deciding split points

In [ ]:
print("Date range:", labeled["order_purchase_timestamp"].min(), "to",
      labeled["order_purchase_timestamp"].max())

labeled.set_index("order_purchase_timestamp").resample("MS").size().plot(
    kind="bar", figsize=(12, 3), title="Orders per month"
)


## 3. Split 70 / 15 / 15 by purchase date

Sorting by `order_purchase_timestamp` and cutting by quantile gives a
70/15/15 split where train is strictly earlier than val, which is
strictly earlier than test.


In [ ]:
labeled_sorted = labeled.sort_values("order_purchase_timestamp").reset_index(drop=True)

n = len(labeled_sorted)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = labeled_sorted.iloc[:train_end].copy()
val = labeled_sorted.iloc[train_end:val_end].copy()
test = labeled_sorted.iloc[val_end:].copy()

for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} n={len(df):6,d}  "
          f"{df['order_purchase_timestamp'].min().date()} -> {df['order_purchase_timestamp'].max().date()}")


## 4. Check label balance in each split

With a time-based split the ratio isn't forced equal — check how much it
actually moves. A small drift is expected and fine; a huge drift is a
sign that delivery performance shifted sharply over the period and is
worth a note for the EDA/modeling notebooks.


In [ ]:
for name, df in [("train", train), ("val", val), ("test", test)]:
    rate = df["is_late"].mean()
    print(f"{name:5s}  late rate = {rate:.2%}  (n={len(df)})")


*(If you had instead chosen a random split, this is the point where you'd
use `train_test_split(..., stratify=labeled[LABEL_COL])` twice to carve
out train/val/test while keeping the label ratio identical across all
three — worth remembering even though we're not using it here.)*


## Artifacts: `train.parquet`, `val.parquet`, `test.parquet`

In [ ]:
train.to_parquet(TRAIN_PATH, index=False)
val.to_parquet(VAL_PATH, index=False)
test.to_parquet(TEST_PATH, index=False)

print("Saved:")
print(" ", TRAIN_PATH, train.shape)
print(" ", VAL_PATH, val.shape)
print(" ", TEST_PATH, test.shape)
